Run `python run_pipeline.py` from the project root (or execute each model
notebook once, in any order) so every `metrics_*.csv` lands in `../results/`,
then run this notebook. It reads those files and writes
`comparison_summary.csv` and `best_model_by_window.csv` back to `../results/`.

In [ ]:
import pandas as pd

In [ ]:
MODEL_FILES = {
    "VECM": "../results/metrics_var_vecm.csv",
    "ARIMA": "../results/metrics_arima.csv",
    "LSTM": "../results/metrics_lstm.csv",
    "TDNN": "../results/metrics_tdnn.csv",
    "FFNN": "../results/metrics_ffnn.csv",
}
WINDOW_COLS = ["W1", "W2", "W3", "W4", "W5"]

In [ ]:
def load_metrics(model: str, path: str) -> pd.DataFrame:
    """Load one model's standardized long-format export: window, target, RMSE, MAE."""
    df = pd.read_csv(path)
    df.insert(0, "model", model)
    return df


long_df = pd.concat([load_metrics(m, p) for m, p in MODEL_FILES.items()], ignore_index=True)
long_df

In [ ]:
# Rows: model x target x metric. Columns: W1..W5.
comparison = (
    long_df
    .melt(id_vars=["model", "window", "target"], value_vars=["RMSE", "MAE"], var_name="metric")
    .pivot_table(index=["model", "target", "metric"], columns="window", values="value")
    .reindex(columns=WINDOW_COLS)
    .reindex(MODEL_FILES.keys(), level="model")
)
comparison.round(4)

In [ ]:
# Best-performing model per window/target, ranked by RMSE.
best_by_window = (
    long_df.loc[long_df.groupby(["window", "target"])["RMSE"].idxmin(), ["window", "target", "model", "RMSE"]]
    .set_index(["window", "target"])
    .sort_index()
)
best_by_window

In [ ]:
# Persist the cross-model comparison so it survives outside this notebook.
comparison.round(4).to_csv("../results/comparison_summary.csv")
best_by_window.to_csv("../results/best_model_by_window.csv")
